<a href="https://colab.research.google.com/github/ritwin82/Neural_Hack_Tralala/blob/main/neural_hack2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install lightgbm catboost xgboost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder

# 1. Load the datasets
train_final = pd.read_csv('train_final.csv')
hidden_history = pd.read_csv('hidden_dist_history.csv')
test_x = pd.read_csv('test_X_q1_2026.csv')

# 2. Combine historical data for training
full_history = pd.concat([train_final, hidden_history], axis=0, ignore_index=True)

# 3. Define target columns and features
targets = ['Persondays_of_Central_Liability_so_far', 'Total_No_of_Works_Takenup']
# Use all numeric features available in the test set
base_features = [col for col in test_x.columns if col not in ['date', 'month', 'district_id', 'state_id']]

# 4. Consistent encoding for district and state IDs
all_districts = pd.concat([full_history['district_id'], test_x['district_id']]).unique()
all_states = pd.concat([full_history['state_id'], test_x['state_id']]).unique()

le_dist = LabelEncoder().fit(all_districts)
le_state = LabelEncoder().fit(all_states)

full_history['district_enc'] = le_dist.transform(full_history['district_id'])
full_history['state_enc'] = le_state.transform(full_history['state_id'])
test_x['district_enc'] = le_dist.transform(test_x['district_id'])
test_x['state_enc'] = le_state.transform(test_x['state_id'])

features = base_features + ['district_enc', 'state_enc']

# 5. Prediction Pipeline
submission = test_x[['district_id', 'date']].copy()

for target in targets:
    # Initialize robust regressor (handles NaNs and non-linear trends)
    model = HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_depth=7,
        l2_regularization=0.1,
        random_state=42
    )

    # Train only on rows where target data exists
    train_data = full_history.dropna(subset=[target])
    model.fit(train_data[features], train_data[target])

    # Predict for Q1 2026
    preds = model.predict(test_x[features])

    # Ensure Persondays (count metric) is not negative
    if target == 'Persondays_of_Central_Liability_so_far':
        preds = np.maximum(preds, 0)

    submission[target] = preds

# 6. Save the results
submission.to_csv('submission.csv', index=False)